In [ ]:
import os, random
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.utils as vutils
import matplotlib.pyplot as plt
import torchvision.utils as vutils
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader
from PIL import Image


# Set random seed for reproducibility
random_seed = 42

random.seed(random_seed)
torch.manual_seed(random_seed)
torch.cuda.manual_seed_all(random_seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


# Hyperparameters and configuration
batch_size = 128
image_size = 32    # 32x32 images
nz = 100           # Size of latent noise vector
ngf = 256          # Generator feature map size for first conv (number of filters)
ndf = 64           # Discriminator base feature map size
num_epochs = 20
lr = 0.0002        # Learning rate for Adam
beta1 = 0.5        # Beta1 hyperparameter for Adam optimizer


# Data preparation: CIFAR-10 training set and test set
transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_dataset = CIFAR10(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)

test_dataset = CIFAR10(root='./data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)


# Generator network: 100-dim noise to 3x32x32 image
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        # [Citation of AI content] - This part is significantly generated by AI.
        # z: (B, nz, 1, 1)
        self.main = nn.Sequential(
            nn.ConvTranspose2d(nz, ngf, 4, 1, 0, bias=False),   # (B, ngf, 4, 4)
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),

            nn.ConvTranspose2d(ngf, ngf // 2, 4, 2, 1, bias=False),  # (B, ngf//2, 8, 8)
            nn.BatchNorm2d(ngf // 2),
            nn.ReLU(True),

            nn.ConvTranspose2d(ngf // 2, ngf // 4, 4, 2, 1, bias=False),  # (B, ngf//4, 16, 16)
            nn.BatchNorm2d(ngf // 4),
            nn.ReLU(True),

            nn.ConvTranspose2d(ngf // 4, 3, 4, 2, 1, bias=False),  # (B, 3, 32, 32)
            nn.Tanh()
        )

    def forward(self, x):
        # z: (B, nz) -> (B, nz, 1, 1)
        x = x.view(-1, nz, 1, 1)
        return self.main(x)


# Discriminator network: 3x32x32 image to probability
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        # x: (B, 3, 32, 32)
        self.main = nn.Sequential(
            nn.Conv2d(3, ndf, 4, 2, 1, bias=False),  # (B, ndf, 16, 16)
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False),  # (B, ndf*2, 8, 8)
            nn.BatchNorm2d(ndf * 2),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf * 2, ndf * 4, 4, 2, 1, bias=False),  # (B, ndf*4, 4, 4)
            nn.BatchNorm2d(ndf * 4),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(ndf * 4, 1, 4, 1, 0, bias=False),  # (B, 1, 1, 1)
            nn.Sigmoid()
        )

    def forward(self, x):
        # Flatten the output to a scalar for each image
        out = self.main(x)
        return out.view(-1)  # flatten to shape (N,) or (N,1)

# Initialize the networks
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
netG = Generator().to(device)
netD = Discriminator().to(device)

# Apply DCGAN weight initialization to Conv and BatchNorm layers:contentReference[oaicite:27]{index=27}
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        # Initialize Conv and ConvTranspose weights
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    if classname.find('BatchNorm') != -1:
        # Initialize BatchNorm weight and bias
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

netG.apply(weights_init)
netD.apply(weights_init)

# Loss function and optimizers
criterion = nn.BCELoss()
optimizerD = optim.Adam(netD.parameters(), lr=lr, betas=(beta1, 0.999))
optimizerG = optim.Adam(netG.parameters(), lr=lr, betas=(beta1, 0.999))

In [ ]:
!pip install torchinfo

In [ ]:
from torchinfo import summary

print("----- DCGAN Generator Summary -----")
summary(netG, input_size=(1, nz, 1, 1))

In [ ]:
print("----- DCGAN Discriminator Summary -----")
summary(netD, input_size=(1, 3, 32, 32))

In [ ]:
# Create fixed noise for image sampling
fixed_noise = torch.randn(64, nz, 1, 1, device=device)

# Create output directory for sample images
os.makedirs("output_images", exist_ok=True)

# Training loop
print("Starting Training...")
for epoch in range(1, num_epochs+1):
    total_loss_D = 0
    total_loss_G = 0

    # [Citation of AI content] - This part is significantly generated by AI.
    for i, (real_images, _) in enumerate(train_loader):
        real_images = real_images.to(device)
        b_size = real_images.size(0)

        real_labels = torch.ones(b_size, device=device)
        fake_labels = torch.zeros(b_size, device=device)

        # (1) Train Discriminator D
        netD.zero_grad()

        real_outputs = netD(real_images)
        real_loss = criterion(real_outputs, real_labels)
        real_loss.backward()

        noise = torch.randn(b_size, nz, 1, 1, device=device)
        fake_images = netG(noise)

        fake_outputs = netD(fake_images.detach())
        fake_loss = criterion(fake_outputs, fake_labels)
        fake_loss.backward()

        loss_D = real_loss + fake_loss
        optimizerD.step()

        # (2) Train Generator G
        netG.zero_grad()
        fake_outputs = netD(fake_images)
        loss_G = criterion(fake_outputs, real_labels)
        loss_G.backward()
        optimizerG.step()

        # Accumulate loss for monitoring
        total_loss_D += loss_D.item()
        total_loss_G += loss_G.item()

    # End of epoch
    avg_loss_D = total_loss_D / len(train_loader)
    avg_loss_G = total_loss_G / len(train_loader)

    print(f"Epoch [{epoch}/{num_epochs}]  Loss_D: {avg_loss_D:.4f}  Loss_G: {avg_loss_G:.4f}")

    # Save sample images at epoch 1, mid (epoch 10), and final (epoch 20)
    if epoch in [1, 10, num_epochs]:
        netG.eval()  # eval mode for stable output (fix BatchNorm)
        with torch.no_grad():
            fake = netG(fixed_noise).detach().cpu()
        # Make a grid of 8x8 (64) images and save
        grid = vutils.make_grid(fake, nrow=8, normalize=True, value_range=(-1, 1))
        vutils.save_image(grid, f"output_images/epoch_{epoch}.png")
        netG.train()  # switch back to train for next epoch (if any)


In [ ]:
!pip install torchmetrics[image]

In [ ]:
# Compute FID score using TorchMetrics (requires torchmetrics library)
try:
    from torchmetrics.image.fid import FrechetInceptionDistance
except ImportError:
    print("Please install torchmetrics (e.g., pip install torchmetrics[image]) to compute FID.")
    exit()

fid = FrechetInceptionDistance(feature=2048).to(device)  # feature dim 2048 for InceptionV3 pool3

# We accumulate real and fake features in the FID metric
netG.eval()  # use eval mode for generator during FID computation

# Loop over real images from test set
# [Citation of AI content] - This part is significantly generated by AI.
real_images_count = 0

for real_images, _ in test_loader:
    real_images = real_images.to(device)

    real_images_scaled = (real_images + 1) / 2 * 255
    real_images_uint8 = real_images_scaled.to(torch.uint8)

    fid.update(real_images_uint8, real=True)
    real_images_count += real_images.size(0)

    if real_images_count >= 5000:  # stop after 5000
        break

# Loop to generate fake images equal to the number of test images
fake_images_count = 0

while fake_images_count < 5000:
    # Generate fake images in batches
    noise = torch.randn(batch_size, nz, 1, 1, device=device)
    fake_images = netG(noise).detach()

    fake_images_scaled = (fake_images + 1) / 2 * 255
    fake_images_uint8 = fake_images_scaled.to(torch.uint8)

    fid.update(fake_images_uint8, real=False)
    fake_images_count += batch_size


# Compute FID
fid_value = fid.compute()
print(f"FID score = {fid_value.item():.4f}")


def display_generated_images(image_dir="output_images", epochs=[1, 10, 20]):
    """
    Displays saved generated image grids from specific epochs for visual inspection.
    Assumes images are saved as PNG files like 'epoch_1.png' etc.
    """
    plt.figure(figsize=(12, 4))
    for i, epoch in enumerate(epochs):
        image_path = os.path.join(image_dir, f"epoch_{epoch}.png")
        if not os.path.exists(image_path):
            print(f"[!] Warning: {image_path} not found.")
            continue

        img = Image.open(image_path)
        plt.subplot(1, len(epochs), i+1)
        plt.imshow(img)
        plt.title(f"Epoch {epoch}")
        plt.axis("off")

    plt.tight_layout()
    plt.show()
display_generated_images(image_dir="output_images", epochs=[1, 10, 20])

In [ ]:
#Load the necessary modules & Set random seed for reproduction
import os
from torch.backends import cudnn
import torch
import numpy as np
import random
import torch.nn as nn
import torch.nn.functional as F
import cv2
from torch import optim
from torchvision import datasets
from torchvision import transforms
from google.colab.patches import cv2_imshow


#set seed value
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
np.random.seed(random_seed)
random.seed(random_seed)

#Set hyper-parameters & Create directories if not exist.
# model hyper-parameters
image_size=32
g_conv_dim=64
d_conv_dim=64
use_labels=True
num_classes=10

# training hyper-parameters
train_iters=1000
batch_size=64
num_workers=2

# misc
mode='train'
model_path='./models'
sample_path='./samples'
mnist_path='./mnist'
svhn_path='./svhn'
download=True

# create directories if not exist
if not os.path.exists(model_path):
    os.makedirs(model_path)
if not os.path.exists(sample_path):
    os.makedirs(sample_path)

#Define `get_loader()` function to transform data and build dataloader for MNIST and SVHN dataset, respectively. And then, we can make each dataloader iterable using `iter()` so that it can be used for train and test phase.
def get_loader():
    """Builds and returns Dataloader for MNIST and SVHN dataset."""
    transform_svhn = transforms.Compose([
        transforms.Resize(image_size),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])
    transform_mnist = transforms.Compose([
        transforms.Resize(image_size),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    svhn = datasets.SVHN(root=svhn_path, split='train', download=download, transform=transform_svhn)
    mnist = datasets.MNIST(root=mnist_path, train=True, download=download, transform=transform_mnist)

    svhn_loader = torch.utils.data.DataLoader(dataset=svhn, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    mnist_loader = torch.utils.data.DataLoader(dataset=mnist, batch_size=batch_size, shuffle=True, num_workers=num_workers)

    return svhn_loader, mnist_loader

In [ ]:
svhn_loader, mnist_loader = get_loader()
svhn_iter = iter(svhn_loader)
mnist_iter = iter(mnist_loader)
iter_per_epoch = min(len(svhn_iter), len(mnist_iter))

# fixed mnist and svhn for sampling

pretty_svhn = [0,1,8,9,12,13,20,21,
        24,25,26,27,28,29,34,35,38,39,
        43,49,50,54,55,58,59,65,66,68,
        69,70,71,74,80,83,84,87,88,89,
        90,91,95,96,97,98,99,102,103,104,
        105,106,107,108,109,110,111,112,113,114,
        115,127,128,129,133,134]
fixed_svhn = torch.tensor(svhn_loader.dataset.data[pretty_svhn]).float().cuda()
fixed_svhn = ((fixed_svhn - fixed_svhn.min()) / (fixed_svhn.max() - fixed_svhn.min()) - 0.5)*2
next(mnist_iter)
fixed_mnist = next(mnist_iter)[0].cuda()
mnist_iter = iter(mnist_loader)

In [ ]:
class Residual_Block(nn.Module):
    def __init__(self, in_dim, mid_dim, out_dim):
        super(Residual_Block, self).__init__()
        self.main = nn.Sequential(
            nn.Conv2d(in_dim, mid_dim, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_dim),
            nn.ReLU(True),

            nn.Conv2d(mid_dim, out_dim, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_dim)
        )
        # [Citation of AI content] - This part is significantly generated by AI.
        if in_dim != out_dim:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_dim, out_dim, kernel_size=1, bias=False),
                nn.BatchNorm2d(out_dim)
            )
        else:
            self.shortcut = nn.Identity()
        self.relu = nn.ReLU(True)

    def forward(self, x):
        out_main = self.main(x)
        out_shortcut = self.shortcut(x)
        out = self.relu(out_main + out_shortcut)
        return out

In [ ]:
#Based on residual block defined above, make a `Generator` class.

class Generator(nn.Module):
    def __init__(self, type, conv_dim=64):
        super(Generator, self).__init__()
        if type == 'mnist_to_svhn':
            in_dim, out_dim = 1, 3
        elif type == 'svhn_to_mnist':
            in_dim, out_dim = 3, 1
        # encoding blocks
        # x: (B, in_dim, 32, 32)
        self.enc = nn.Sequential(
            # (B, conv_dim, 16, 16)
            nn.Conv2d(in_dim, conv_dim, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            # (B, conv_dim*2, 8, 8)
            nn.Conv2d(conv_dim, conv_dim * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(conv_dim * 2),
            nn.LeakyReLU(0.2, inplace=True),

            # (B, conv_dim*4, 4, 4)
            nn.Conv2d(conv_dim * 2, conv_dim * 4, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(conv_dim * 4),
            nn.LeakyReLU(0.2, inplace=True)
        )
        # residual blocks
        self.res = nn.Sequential(
            Residual_Block(conv_dim * 4, conv_dim * 4, conv_dim * 4),
            Residual_Block(conv_dim * 4, conv_dim * 4, conv_dim * 4),
            Residual_Block(conv_dim * 4, conv_dim * 4, conv_dim * 4),
            Residual_Block(conv_dim * 4, conv_dim * 4, conv_dim * 4),
            Residual_Block(conv_dim * 4, conv_dim * 4, conv_dim * 4),
            Residual_Block(conv_dim * 4, conv_dim * 4, conv_dim * 4)
        )
        # decoding blocks
        # x: (B, conv_dim*4, 4, 4)
        self.dec = nn.Sequential(
            # (B, conv_dim*2, 8, 8)
            nn.ConvTranspose2d(conv_dim * 4, conv_dim * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(conv_dim * 2),
            nn.ReLU(True),
            # (B, conv_dim, 16, 16)
            nn.ConvTranspose2d(conv_dim * 2, conv_dim, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(conv_dim),
            nn.ReLU(True),
            # (B, out_dim, 32, 32)
            nn.ConvTranspose2d(conv_dim, out_dim, kernel_size=4, stride=2, padding=1, bias=False),
            nn.Tanh()
        )
        # other layers

    def forward(self, x):
        x = self.enc(x)
        x = self.res(x)
        out = self.dec(x)
        return out

In [ ]:
#`Discriminator` class which has a total of 11 classes (classes 0 through 9 are for real images and class 10 is for fake images).
class Discriminator(nn.Module):
    def __init__(self, type, conv_dim=64):
        super(Discriminator, self).__init__()
        if type == 'mnist':
            in_dim = 1
        elif type == 'svhn':
            in_dim = 3
        # in_dim, out_dim = 1, 1

        # learnable layers
        self.enc = nn.Sequential(
            # (B, conv_dim, 16, 16)
            nn.Conv2d(in_dim, conv_dim, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            # (B, conv_dim*2, 8, 8)
            nn.Conv2d(conv_dim, conv_dim * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(conv_dim * 2),
            nn.LeakyReLU(0.2, inplace=True),

            # (B, conv_dim*4, 4, 4)
            nn.Conv2d(conv_dim * 2, conv_dim * 4, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(conv_dim * 4),
            nn.LeakyReLU(0.2, inplace=True)
        )

        # self.fc = nn.Linear(conv_dim * 2, 11) # Note that classes 0 through 9 are for real images and class 10 is for fake images.
        self.fc = nn.Linear(conv_dim * 4 * 4 * 4, 11) # Note that classes 0 through 9 are for real images and class 10 is for fake images.

        # other functions
        self.relu = nn.ReLU()
        self.downsample = nn.MaxPool2d(kernel_size=3, padding= 1, stride=2)

    def forward(self, x):
        x = self.enc(x)
        x = x.view(x.size(0), -1)
        out = self.fc(x)    # (B, 11)
        return out

In [ ]:
#Define the `optimize_model` function for optimizing networks
def optimize_model(optimizer, loss):
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

In [ ]:
#Next, define some auxiliary functions that can be used to merge images, show log and save images: `merge_images()`, `show_log()`, `save_img()`.
def merge_images(sources, targets):
    _, _, h, w = sources.shape
    row = int(np.sqrt(batch_size))
    merged = np.zeros([3, row * h, row * w * 2])
    for idx, (s, t) in enumerate(zip(sources, targets)):
        i = idx // row
        j = idx % row
        merged[:, i * h:(i + 1) * h, (j * 2) * h:(j * 2 + 1) * h] = s
        merged[:, i * h:(i + 1) * h, (j * 2 + 1) * h:(j * 2 + 2) * h] = t
    return merged.transpose(1, 2, 0)


def show_log(step, train_iters, D_ms_losses, D_sm_losses, G_msm_losses, G_ses_losses):
    D_ms_avg = np.array(D_ms_losses).mean()
    D_sm_avg = np.array(D_sm_losses).mean()
    G_msm_avg = np.array(G_msm_losses).mean()
    G_sms_avg = np.array(G_ses_losses).mean()

    print('Step [%d/%d], D_ms_loss: %.4f, D_sm_loss: %.4f, G_msm_loss: %.4f, G_sms_loss: %.4f'
    % (step + 1, train_iters, D_ms_avg, D_sm_avg, G_msm_avg, G_sms_avg))


def save_img(step, fixed_mnist, fixed_svhn, netG, netF, m_data, s_data, sample_path):
    fake_svhn = netG(fixed_mnist)
    fake_mnist = netF(fixed_svhn)

    mnist, fake_mnist = fixed_mnist.detach().cpu().numpy(), fake_mnist.detach().cpu().numpy()
    svhn, fake_svhn = fixed_svhn.detach().cpu().numpy(), fake_svhn.detach().cpu().numpy()

    merged = merge_images(mnist, fake_svhn)
    merged = np.interp(merged, (-1, 1), (0, 255))
    path = os.path.join(sample_path, 'sample-%d-m-s.png' % (step + 1))
    cv2.imwrite(path, merged)
    print('saved %s' % path)

    merged = merge_images(svhn, fake_mnist)
    merged = np.interp(merged, (-1, 1), (0, 255))
    path = os.path.join(sample_path, 'sample-%d-s-m.png' % (step + 1))
    cv2.imwrite(path, merged)
    print('saved %s' % path)

In [ ]:
import itertools

# Build models and their optimizers

#Generator
netG = Generator('mnist_to_svhn', conv_dim=g_conv_dim).to(device)
netF = Generator('svhn_to_mnist', conv_dim=g_conv_dim).to(device)

netG.apply(weights_init)
netF.apply(weights_init)

#Discriminator
Dx = Discriminator('mnist', conv_dim=d_conv_dim).to(device)
Dy = Discriminator('svhn', conv_dim=d_conv_dim).to(device)

Dx.apply(weights_init)
Dy.apply(weights_init)

# [Citation of AI content] - This part is significantly generated by AI.
#Optimizer
genOptimizer = optim.Adam(itertools.chain(netG.parameters(), netF.parameters()), lr=lr, betas=(0.5, 0.999))
disOptimizer = optim.Adam(itertools.chain(Dx.parameters(), Dy.parameters()), lr=lr, betas=(0.5, 0.999))

In [ ]:
print("----- CycleGAN Generator (MNIST to SVHN) Summary -----")
summary(netG, input_size=(1, 1, 32, 32))

In [ ]:
print("----- CycleGAN Discriminator (MNIST) Summary -----")
summary(Dx, (1, 1, 32, 32))

In [ ]:
def Train_D_ms(Dx, Dy, netG, m_data, m_label):
    '''
    # parameter:
    Dx, Dy: Discriminator for source domain(X, MNIST) and target domain(Y, SVHN), respectively.
    G: Generator which transfers X to Y
    m_data: source domain data (i.e., MNIST real images)
    m_label: source domain label (from 0 to 9)

    Train two discriminators(Dx, Dy) in mnist-svhn cycle.
    Please refer to the above figure that represents the path from X to Y and equation (1),(3).
    Unlike the conventional GAN, here, the discriminator classifies 11 classes. You have to use 1) the original labels(from 0 to 9) when training the discriminator with real images, and 2) the fake label(10) when training the discriminator with fake(generated) images. You should make the fake label for training.
    So, "torch.nn.functional.cross_entropy()" will be used for calculating losses.


    Return:
    D_ms_loss.item(): item of the loss
    '''

    # train mnist-svhn cycle

    # 1) Compute the loss of Dx (Dx_loss) with real images
    ### write your code --------------------------------------
    out_x = Dx(m_data)
    Dx_loss = F.cross_entropy(out_x, m_label)
    ### --------------------------------------------------------

    # 2) Compute the loss of Dy (Dy_loss) with fake images
    ### write your code --------------------------------------
    fake_s = netG(m_data).detach()
    out_y = Dy(fake_s)
    fake_label = torch.full((m_label.size(0),), 10, dtype=torch.long, device=device)
    Dy_loss = F.cross_entropy(out_y, fake_label)
    ### --------------------------------------------------------

    D_ms_loss = Dx_loss + Dy_loss
    optimize_model(disOptimizer, D_ms_loss)
    return D_ms_loss.item()

In [ ]:
def Train_D_sm(Dx, Dy, netF, s_data, s_label):
    '''
    # parameter:
    Dx, Dy: Discriminator for source domain(X, MNIST) and target domain(Y, SVHN), respectively.
    F: Generator which transfers Y to X
    s_data: target domain data (i.e., SVHN real images)
    s_label: target domain label (from 0 to 9)

    Train discriminators in svhn-mnist cycle.
    Please refer to the above figure that represents the path from Y to X and equation (1),(3).
    Unlike the conventional GAN, here, the discriminator classifies 11 classes. You have to use 1) the original labels(from 0 to 9) when training the discriminator with real images, and 2) the fake label(10) when training the discriminator with fake(generated) images. You should make the fake label for training.
    So, "torch.nn.functional.cross_entropy()" will be used for calculating losses.

    Return:
    D_sm_loss.item(): item of the loss
    '''

    # train svhn-mnist cycle

    # 1) Compute the loss of Dy (Dy_loss) with real images
    ### write your code --------------------------------------
    out_y = Dy(s_data)
    Dy_loss = F.cross_entropy(out_y, s_label)
    ### --------------------------------------------------------

    # 2) Compute the loss of Dx (Dx_loss) with fake images
    ### write your code --------------------------------------
    fake_m = netF(s_data).detach()
    out_x = Dx(fake_m)
    fake_label = torch.full((s_label.size(0),), 10, dtype=torch.long, device=device)
    Dx_loss = F.cross_entropy(out_x, fake_label)
    ### --------------------------------------------------------

    D_sm_loss = Dx_loss + Dy_loss
    optimize_model(disOptimizer, D_sm_loss)
    return D_sm_loss.item()

In [ ]:
# train mnist-svhn-mnist cycle
def Train_G_msm(Dy, netG, netF, m_data, m_label):
    '''
    # parameter:
    Dy: Discriminator for target domain(Y, SVHN).
    G, F: Generator for X to Y and Y to X.
    m_data: source domain data (i.e., MNIST real images)
    m_label: source domain label (from 0 to 9)

    Train mnist-svhn-mnist cycle.
    Please refer to the above figure that represents the path from X to Y to X and equation (1),(5).
    Note that you have to use original labels(from 0 to 9) when training the generator.
    So, "torch.nn.functional.cross_entropy()" will be used for calculating losses.
    Also, this function has to include cycle consistency loss(i.e. Lcyc) with mean squared error loss.

    Return:
    G_msm_loss.item(): item of the loss
    '''

    # Compute the generator G loss (G_loss) with fake images
    ### write your code--------------------------------------
    fake_s = netG(m_data)
    out_y = Dy(fake_s)
    G_loss = F.cross_entropy(out_y, m_label)
    ### --------------------------------------------------------

    # Compute the cycle consistency loss (Lcyc)
    ### write your code--------------------------------------
    recon_m = netF(fake_s)
    Lcyc = F.mse_loss(recon_m, m_data)
    ### --------------------------------------------------------

    G_msm_loss = G_loss + 100 * Lcyc
    optimize_model(genOptimizer, G_msm_loss)
    return G_msm_loss.item()

In [ ]:
# train svhn-mnist-svhn cycle
def Train_G_sms(Dx, netG, netF, s_data, s_label):
    '''
    # parameter:
    Dx: Discriminator for source domain(X, MNIST).
    G, F: Generator for X to Y and Y to X.
    s_data: target domain data (i.e., SVHN real images)
    s_label: target domain label (from 0 to 9)

    Train svhn-mnist-svhn cycle.
    Please refer to the above figure that represents the path from Y to X to Y and equation (3),(5).
    Note that you have to use original labels(from 0 to 9) when training the generator.
    So, "torch.nn.functional.cross_entropy()" will be used for calculating losses.
    Also, this function has to include cycle consistency loss(i.e. Lcyc) with mean squared error loss.

    Return:
    G_sms_loss.item(): item of the loss
    '''

    # Compute the generator F loss (F_loss) with fake images
    ### write your code --------------------------------------
    fake_m = netF(s_data)
    out_x = Dx(fake_m)
    F_loss = F.cross_entropy(out_x, s_label)
    ### --------------------------------------------------------

    # Compute the cycle consistency loss (Lcyc)
    ### write your code --------------------------------------
    recon_s = netG(fake_m)
    Lcyc = F.mse_loss(recon_s, s_data)
    ### --------------------------------------------------------

    G_sms_loss = F_loss + 100 * Lcyc
    optimize_model(genOptimizer, G_sms_loss)
    return G_sms_loss.item()

In [ ]:
D_ms_losses, D_sm_losses, G_msm_losses, G_sms_losses = [], [], [], []

for step in range(train_iters + 1):
    # reset data_iter for each epoch
    if (step + 1) % iter_per_epoch == 0:
        mnist_iter = iter(mnist_loader)
        svhn_iter = iter(svhn_loader)

    # load svhn and mnist dataset
    s_data, s_label = next(svhn_iter)
    s_data, s_label = s_data.cuda(), s_label.cuda().long().squeeze()
    m_data, m_label = next(mnist_iter)
    m_data, m_label = m_data.cuda(), m_label.cuda()

    D_ms_losses.append(Train_D_ms(Dx, Dy, netG, m_data, m_label))
    D_sm_losses.append(Train_D_sm(Dx, Dy, netF, s_data, s_label))
    G_msm_losses.append(Train_G_msm(Dy, netG, netF, m_data, m_label))
    G_sms_losses.append(Train_G_sms(Dx, netG, netF, s_data, s_label))

    # print the log info
    if (step + 1) % 100 == 0:
        show_log(step, train_iters, D_ms_losses, D_sm_losses, G_msm_losses, G_sms_losses)

    # save the sampled images
    if (step + 1) == 1000:
        save_img(step, fixed_mnist, fixed_svhn, netG, netF, m_data, s_data, sample_path)

In [ ]:
# [Citation of AI content] - This part is significantly generated by AI.
netG.eval()
netF.eval()

original_mnist = fixed_mnist
original_svhn = fixed_svhn

with torch.no_grad():
    # MNIST -> SVHN -> MNIST Cycle
    fake_svhn = netG(original_mnist)
    recon_mnist = netF(fake_svhn)

    # SVHN -> MNIST -> SVHN Cycle
    fake_mnist = netF(original_svhn)
    recon_svhn = netG(fake_mnist)

def denorm(x):
    out = (x + 1) / 2
    return out.clamp(0, 1)

#-----------
print('MNIST_to_SVHN')

result_m_s = torch.cat([
    denorm(original_mnist[:16]).repeat(1, 3, 1, 1),
    denorm(fake_svhn[:16]),
    denorm(recon_mnist[:16]).repeat(1, 3, 1, 1)
], dim=0)

grid_m_s = vutils.make_grid(result_m_s, nrow=16)
plt.figure(figsize=(15, 5))
plt.imshow(np.transpose(grid_m_s.cpu().numpy(), (1, 2, 0)))
plt.title('Original MNIST -> Fake SVHN -> Reconstructed MNIST')
plt.axis('off')
plt.show()

#-----------
print('\nSVHN_to_MNIST')

result_s_m = torch.cat([
    denorm(original_svhn[:16]),
    denorm(fake_mnist[:16]).repeat(1, 3, 1, 1),
    denorm(recon_svhn[:16])
], dim=0)

grid_s_m = vutils.make_grid(result_s_m, nrow=16)
plt.figure(figsize=(15, 5))
plt.imshow(np.transpose(grid_s_m.cpu().numpy(), (1, 2, 0)))
plt.title('Original SVHN -> Fake MNIST -> Reconstructed SVHN')
plt.axis('off')
plt.show()

#-----------
# Cycle Consistency Check: L1 and L2 losses
# L1 Loss
l1_loss_m = F.l1_loss(recon_mnist, original_mnist)
l1_loss_s = F.l1_loss(recon_svhn, original_svhn)
print(f'\nCycle Consistency L1 Loss (MNIST): {l1_loss_m.item():.4f}')
print(f'Cycle Consistency L1 Loss (SVHN): {l1_loss_s.item():.4f}')

# L2 Loss
l2_loss_m = F.mse_loss(recon_mnist, original_mnist)
l2_loss_s = F.mse_loss(recon_svhn, original_svhn)
print(f'\nCycle Consistency L2 Loss (MNIST): {l2_loss_m.item():.4f}')
print(f'Cycle Consistency L2 Loss (SVHN): {l2_loss_s.item():.4f}')